# Quantum Chemistry in a Nutshell

In quantum chemistry, we generally want to solve the electronic Schrödinger equation for a molecule whose Hamiltonian can be written in second quantization as: 

$$
\hat{H} = \sum_{pq} h_{pq} a_p^\dagger a_q + \frac{1}{2} \sum_{pqrs} g_{pqrs} a_p^\dagger a_q^\dagger a_s a_r
$$

This Hamiltonian consists of two basic kinds of ingredients: the Fermionic creation and annihilation operators, $a_p^\dagger$ and $a_q$, which create and annihilate electrons in single particle states (orbitals), and the one-electron integrals $h_{pq}$ and two-electron integrals $g_{pqrs}$, which encode the kinetic energy of the electrons and their interactions with each other and with the nuclei. Written in this form, we assume that the orbitals that we use are orthonormal. 


The first step in defining the Hamiltonian is to choose a basis set of single particle states which we will use to represent the molecular orbitals. The choice of basis set is crucial, as it determines the accuracy and computational cost of our calculations. Common choices include atom centered Gaussian basis sets, plane wave basis sets, and basis functions derived from numerical solutions of the atomic Schrödinger equation and represented on a grid. In our studies we will focus on Gaussian basis sets, which are widely used in quantum chemistry due to their computational efficiency and flexibility.

A general cartesian Gaussian basis function centered at a point $\mathbf{A} = (A_x, A_y, A_z)$ can be written as:   

$$  
g_{lmn}(\mathbf{r}) = (x - A_x)^{l} (y - A_y)^{m} (z - A_z)^{n} e^{-\alpha |\mathbf{r} - \mathbf{A}|^2}
$$

Such basis functions do not form an orthonormal set, and thus we need first to orghogonalize them before we can use them to represent the molecular orbitals. The orthogonalization process involves computing the overlap matrix $S_{pq} = \langle g_p | g_q \rangle$ and then applying a transformation to obtain an orthonormal set of basis functions. Thus the first step in quantum chemical calculations is to compute the overlap matrix and perform the orthogonalization. 


Once we have orthonormal orbitals we can proceed further and compute one electron integrals involving the kinetic energy and nuclear attraction integrals which are defined as:    

$$
T_{pq} = \langle g_p | -\frac{1}{2} \nabla^2 | g_q \rangle
$$

$$
V_{pq} = \langle g_p | -\sum_A \frac{Z_A}{|\mathbf{r} - \mathbf{R}_A|} | g_q \rangle
$$

The two electron integrals are defined as:

$$
g_{pqrs} = \langle g_p g_q | \frac{1}{|\mathbf{r}_1 - \mathbf{r}_2|} | g_r g_s \rangle
$$  

With one and two electron integrals in hand, we can then construct the Hamiltonian and proceed to solve the electronic Schrödinger equation using various methods such as Hartree-Fock, Configuration Interaction, Coupled Cluster, or Quantum Monte Carlo. Each of these methods has its own advantages and disadvantages in terms of accuracy and computational cost, and the choice of method depends on the specific problem at hand.   

In general, most wavefunction based methods involve constructing a trial wavefunction as a linear combination of Slater determinants, which are antisymmetrized products of single particle orbitals and can be represented in the second quantized formalism as:

$$
|\Psi\rangle = \sum_{i,j,k,\ldots} c_{i,j,k,\ldots} a^\dagger_i a^\dagger_j a^\dagger_k \ldots |vac\rangle
$$  

where $|vac\rangle$ is the vacuum state with no electrons, the operators $a^\dagger_i$ create electrons in the single particle orbitals, and the coefficients $c_{i,j,k,\ldots}$ are variational parameters that we optimize to minimize the energy. If only a single determinant is used, we have the Hartree-Fock method, while if we include multiple determinants we can capture electron correlation effects and obtain more accurate results. Different methods differ in how they select the determinants to include in the expansion and how they optimize the coefficients. For example, Configuration Interaction (CI) includes all possible determinants up to a certain excitation level, while Coupled Cluster (CC) includes a specific subset of determinants that are generated by applying an exponential cluster operator to a reference determinant. Quantum Monte Carlo (QMC) methods, on the other hand, use stochastic sampling to evaluate the energy and other properties of the system without explicitly constructing the wavefunction. However, all these methods rely on the same underlying Hamiltonian and the same set of one and two electron integrals, which are the fundamental building blocks of quantum chemical calculations and thus we will develop and implement algorithms to compute these integrals first. 




  We will study in detail the properties of the Fermionic operators and focus for now on the integrals, which are defined as: 

$$
h_{pq} = \int d\mathbf{r} \, \phi_p^*(\mathbf{r}) \left( -\frac{1}{2} \nabla^2 - \sum_A \frac{Z_A}{|\mathbf{r} - \mathbf{R}_A|} \right) \phi_q(\mathbf{r})
$$

$$
g_{pqrs} = \int d\mathbf{r}_1 d\mathbf{r}_2 \, \phi_p^*(\mathbf{r}_1) \phi_q^*(\mathbf{r}_2) \frac{1}{|\mathbf{r}_1 - \mathbf{r}_2|} \phi_r(\mathbf{r}_1) \phi_s(\mathbf{r}_2)
$$

Here, $\phi_p(\mathbf{r})$ are the single particle orbitals, which are typically obtained from a mean-field calculation such as Hartree-Fock. The one-electron integrals $h_{pq}$ represent the kinetic energy of the electrons and their interactions with the nuclei, while the two-electron integrals $g_{pqrs}$ represent the electron-electron repulsion. These integrals are the building blocks of the electronic structure problem, and their accurate computation is crucial for obtaining reliable results in quantum chemistry. In the next sections, we will discuss how to compute these integrals and how to use them to solve the electronic Schrödinger equation using various methods such as Hartree-Fock, Configuration Interaction, and Coupled Cluster.

## Contracted Gaussian Basis Sets

The basis functions we have defined so far are primitive Gaussian functions, which are not very efficient for representing molecular orbitals. To improve the efficiency of our calculations, we typically use contracted Gaussian basis sets, which are linear combinations of primitive Gaussians. A contracted Gaussian function can be written as:  

$$
\chi(\mathbf{r}) = \sum_{i} c_i g_i(\mathbf{r})
$$  

where $c_i$ are the contraction coefficients and $g_i(\mathbf{r})$ are the primitive Gaussian functions of the form given above. The contraction coefficients are typically obtained from atomic calculations and are designed to reproduce the shape of atomic orbitals. By using contracted Gaussian basis sets, we can achieve a good balance between accuracy and computational cost, as they allow us to represent molecular orbitals with fewer basis functions compared to using primitive Gaussians alone. In practice, there are many different contracted Gaussian basis sets available, such as STO-3G, 6-31G, cc-pVDZ, and many others, each with its own set of contraction coefficients and designed for different levels of accuracy and computational cost. 

Since they are used to represent atomic orbitals, gaussian basis sets are usually organized in shells, where each shell corresponds to a specific angular momentum quantum number. For example, an s-shell corresponds to $l=0$, a p-shell corresponds to $l=1$, a d-shell corresponds to $l=2$, and so on. Each shell can contain multiple basis functions, which are typically contracted Gaussian functions as described above. The number of basis functions in each shell depends on the specific basis set being used and the level of accuracy desired. For example, the STO-3G basis set contains one s-shell and one p-shell for each atom, while the 6-31G basis set contains two s-shells and one p-shell for each atom. The choice of basis set and the number of shells used can have a significant impact on the accuracy and computational cost of quantum chemical calculations, and thus it is important to choose an appropriate basis set for the problem at hand.